# LeetCode 1544 — Make The String Great bằng Python

**Link đề:** https://leetcode.com/problems/make-the-string-great/
**Độ khó:** Easy · **Chủ đề:** String, Stack

```python
class Solution:
    def makeGood(self, s: str) -> str:
        ...
```

### Mục tiêu học
1. Hiểu cặp "xấu": hai ký tự **kề nhau**, **cùng chữ cái** nhưng **khác hoa/thường** (`aA`, `Bb`).
2. Thành thạo các hàm ký tự: `str.lower()`, `str.upper()`, `str.swapcase()`, `ord()`, `chr()`.
3. Hiểu cấu trúc dữ liệu **stack (ngăn xếp)** trên list Python: `append`, `pop`, `stack[-1]`.
4. Đi từ mô phỏng lặp `O(n²)` → stack `O(n)` → hai con trỏ ghi đè tại chỗ.
5. Trace stack từng bước, chứng minh vì sao thứ tự xóa không ảnh hưởng kết quả, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — xóa các cặp "xấu" cho đến khi hết

Cho chuỗi `s` gồm chữ cái tiếng Anh hoa và thường. Chuỗi **tốt** là chuỗi không có hai ký tự kề nhau `s[i]`, `s[i+1]` mà một ký tự là chữ thường và ký tự kia là **cùng chữ đó** viết hoa (hoặc ngược lại).

Mỗi bước, ta được chọn một cặp kề nhau "xấu" và **xóa cả hai**. Lặp đến khi chuỗi tốt. Trả về chuỗi tốt (đề đảm bảo đáp án duy nhất). Chuỗi rỗng cũng là tốt.

| Ví dụ | `s` | Output | Diễn giải |
|---|---|---|---|
| 1 | `"leEeetcode"` | `"leetcode"` | xóa `eE` (hoặc `Ee`) |
| 2 | `"abBAcC"` | `""` | `abBAcC → aAcC → cC → ""` |
| 3 | `"s"` | `"s"` | không có cặp nào |

**Constraints:** `1 <= s.length <= 100`, `s` chỉ gồm chữ cái hoa/thường.

Câu hỏi gợi mở:
- `"aa"` và `"AA"` có phải cặp xấu không? (Không — phải **khác** hoa/thường.)
- `"ab"` + `"B"`? `bB` là cặp xấu; xóa xong `a` đứng cạnh ký tự tiếp theo — **cặp mới** có thể xuất hiện!

In [1]:
for pair in ["aA", "Aa", "aa", "AA", "aB", "bB"]:
    x, y = pair
    bad = x != y and x.lower() == y.lower()
    print(f"{pair!r}: cặp xấu? {bad}")

'aA': cặp xấu? True
'Aa': cặp xấu? True
'aa': cặp xấu? False
'AA': cặp xấu? False
'aB': cặp xấu? False
'bB': cặp xấu? True


`x, y = pair` **unpack** chuỗi 2 ký tự thành 2 biến. `!r` trong f-string in kèm dấu nháy, dễ nhìn ký tự.

## 2. Nền tảng Python cần dùng

### 2.1. `lower()`, `upper()`, `swapcase()`

Các phương thức chuỗi trả về **chuỗi mới** (không sửa chuỗi cũ):
- `"a".upper()` → `"A"`; `"A".lower()` → `"a"`.
- `"a".swapcase()` → `"A"`; `"A".swapcase()` → `"a"` — đổi hoa ↔ thường.

In [2]:
ch = "e"
print(ch.upper(), "E".lower(), ch.swapcase(), "E".swapcase())
print("Điều kiện xấu gọn nhất: x.swapcase() == y")
for x, y in [("e", "E"), ("E", "e"), ("e", "e"), ("e", "F")]:
    print(x, y, x.swapcase() == y)

E e E e
Điều kiện xấu gọn nhất: x.swapcase() == y
e E True
E e True
e e False
e F False


### 2.2. `ord()` và `chr()` — mã số của ký tự

`ord(ch)` cho mã Unicode/ASCII: `'A' = 65 ... 'Z' = 90`, `'a' = 97 ... 'z' = 122`. Chữ thường và hoa **cùng chữ** cách nhau đúng **32**. Vì vậy cặp xấu ⇔ `abs(ord(x) - ord(y)) == 32` (với input chỉ toàn chữ cái).

In [3]:
for ch in "AaEezZ":
    print(ch, ord(ch))
print("ord('a') - ord('A') =", ord("a") - ord("A"))
print("chr(101) =", chr(101))

A 65
a 97
E 69
e 101
z 122
Z 90
ord('a') - ord('A') = 32
chr(101) = e


### 2.3. Stack (ngăn xếp) bằng list

Stack là cấu trúc **vào sau ra trước** (LIFO — Last In, First Out), như chồng đĩa: chỉ đặt/lấy ở **đỉnh**.

| Thao tác | Python | Ý nghĩa |
|---|---|---|
| đẩy vào | `stack.append(x)` | đặt lên đỉnh — `O(1)` |
| lấy ra | `stack.pop()` | bỏ và trả phần tử đỉnh — `O(1)` |
| xem đỉnh | `stack[-1]` | không bỏ ra |
| rỗng? | `not stack` | `True` nếu rỗng |

In [4]:
stack = []
stack.append("a"); stack.append("b"); stack.append("c")
print("stack:", stack, "| đỉnh:", stack[-1])
top = stack.pop()
print("pop ra:", top, "| còn:", stack)
print("rỗng?", not stack)
stack.clear()
print("sau clear, rỗng?", not stack)

stack: ['a', 'b', 'c'] | đỉnh: c
pop ra: c | còn: ['a', 'b']
rỗng? False
sau clear, rỗng? True


`stack[-1]` trên list **rỗng** gây `IndexError` — luôn kiểm tra `stack` khác rỗng trước. Python viết `if stack and stack[-1] ...`: nhờ `and` ngắn mạch, nếu `stack` rỗng thì không truy cập `stack[-1]`.

In [5]:
empty = []
try:
    empty[-1]
except IndexError as e:
    print("Lỗi:", e)
print("An toàn:", bool(empty and empty[-1] == "x"))

Lỗi: list index out of range
An toàn: False


### 2.4. Cắt chuỗi `s[:i] + s[i+2:]`

Xóa 2 ký tự tại vị trí `i, i+1` bằng cách ghép phần trước và phần sau. Mỗi lần tạo chuỗi mới tốn `O(n)`.

In [6]:
s = "leEeetcode"
i = 1
print(s[:i], "+", s[i + 2:], "=", s[:i] + s[i + 2:])

l + eetcode = leetcode


## 3. Cách 1 — Mô phỏng đúng theo đề

Làm y như đề nói: tìm một cặp xấu, xóa, lặp lại cho đến khi không còn cặp nào. Đây là "baseline" dễ hiểu nhất và chính là gợi ý 2 của LeetCode.

In [7]:
def is_bad(x: str, y: str) -> bool:
    return x != y and x.lower() == y.lower()

def make_good_simulate(s: str) -> str:
    changed = True
    while changed:
        changed = False
        for i in range(len(s) - 1):
            if is_bad(s[i], s[i + 1]):
                s = s[:i] + s[i + 2:]       # xóa cặp
                changed = True
                break                        # chuỗi đã đổi → quét lại từ đầu
    return s

print(repr(make_good_simulate("leEeetcode")))
print(repr(make_good_simulate("abBAcC")))
print(repr(make_good_simulate("s")))

'leetcode'
''
's'


`repr("")` in ra `''` — cách tốt để **nhìn thấy** chuỗi rỗng (nếu `print("")` chỉ ra dòng trống).

### 3.1. Chi phí của mô phỏng

Mỗi lần xóa: quét tìm `O(n)` + tạo chuỗi mới `O(n)`. Có tới `n/2` lần xóa → **`O(n²)`**. Tệ nhất là chuỗi dạng `"abc...CBA"` (cặp xấu chỉ lộ ra ở **giữa**, phải xóa từ trong ra ngoài):

In [8]:
def count_scan_steps(s):
    steps = 0; changed = True
    while changed:
        changed = False
        for i in range(len(s) - 1):
            steps += 1
            if is_bad(s[i], s[i + 1]):
                s = s[:i] + s[i + 2:]; changed = True; break
    return steps

import string
for m in [5, 10, 20, 26]:
    t = string.ascii_lowercase[:m] + string.ascii_uppercase[:m][::-1]
    print(f"len={len(t):>2}: số bước quét = {count_scan_steps(t)}")

len=10: số bước quét = 15
len=20: số bước quét = 55
len=40: số bước quét = 210
len=52: số bước quét = 351


`string.ascii_lowercase` là `"abc...z"`. Số bước tăng gần như **bình phương** theo độ dài.

## 4. Nút thắt và insight — chỉ cần nhìn "ký tự cuối đã giữ"

**Nút thắt:** sau mỗi lần xóa, mô phỏng quét lại **từ đầu**, dù phần đầu đã "sạch".

**Insight:** khi đọc từ trái sang phải, giả sử phần đã đọc đã được làm sạch thành chuỗi tốt `kept`. Ký tự mới `ch` chỉ có thể tạo cặp xấu với **ký tự cuối** của `kept`:
- Nếu `kept[-1]` và `ch` xấu → xóa `kept[-1]` (và không thêm `ch`). Ký tự cuối mới của `kept` lại sẵn sàng ghép với ký tự kế tiếp — đây chính là phản ứng **dây chuyền** như `abBA`.
- Ngược lại → thêm `ch` vào cuối.

"Chỉ thao tác ở cuối" = **stack**!

Thử bằng tay với `"abBA"`:

In [9]:
kept = []
for ch in "abBA":
    if kept and is_bad(kept[-1], ch):
        print(f"'{ch}' khử '{kept[-1]}'", end=" → ")
        kept.pop()
    else:
        print(f"đẩy '{ch}'", end=" → ")
        kept.append(ch)
    print(kept)

đẩy 'a' → ['a']
đẩy 'b' → ['a', 'b']
'B' khử 'b' → ['a']
'A' khử 'a' → []


`print(..., end=" → ")` thay ký tự xuống dòng mặc định bằng `" → "`, để hai lệnh print nằm cùng một dòng.

## 5. Lời giải chính — Stack

### 5.1. Từng đoạn
1. `stack = []` — giữ phần đã làm sạch.
2. Với mỗi `ch` trong `s`:
   - `if stack and stack[-1].swapcase() == ch:` → `stack.pop()`.
   - `else:` → `stack.append(ch)`.
3. `return "".join(stack)`.

Điều kiện `stack[-1].swapcase() == ch` gói gọn cả "cùng chữ" và "khác hoa/thường": `swapcase` của `'e'` là `'E'`, không bao giờ bằng chính `'e'`.

### 5.2. Ghép lại

In [10]:
def make_good_stack(s: str) -> str:
    stack = []
    for ch in s:
        if stack and stack[-1].swapcase() == ch:
            stack.pop()                 # khử cặp xấu với đỉnh
        else:
            stack.append(ch)
    return "".join(stack)

print(repr(make_good_stack("leEeetcode")))
print(repr(make_good_stack("abBAcC")))
print(repr(make_good_stack("s")))

'leetcode'
''
's'


## 6. TRACE stack từng bước

`s = "abBAcC"`:

| bước | `ch` | đỉnh trước | xấu? | hành động | stack sau |
|---|---|---|---|---|---|
| 1 | a | — | — | push | `a` |
| 2 | b | a | ✗ | push | `a b` |
| 3 | B | b | ✓ | pop | `a` |
| 4 | A | a | ✓ | pop | (rỗng) |
| 5 | c | — | — | push | `c` |
| 6 | C | c | ✓ | pop | (rỗng) |

Kết quả `""`. Bước 4 là **phản ứng dây chuyền**: sau khi `bB` bị khử, `a` lộ ra và khử tiếp với `A`.

In [11]:
def trace_stack(s):
    stack = []
    print(f"s = {s!r}")
    for step, ch in enumerate(s, start=1):
        top = stack[-1] if stack else None
        if stack and stack[-1].swapcase() == ch:
            stack.pop()
            act = f"POP  ('{top}' + '{ch}' là cặp xấu)"
        else:
            stack.append(ch)
            act = "PUSH"
        print(f"bước {step}: ch='{ch}', đỉnh trước={top!s:>4} → {act:<28} stack={''.join(stack)!r}")
    return "".join(stack)

assert trace_stack("abBAcC") == ""
print()
assert trace_stack("leEeetcode") == "leetcode"

s = 'abBAcC'
bước 1: ch='a', đỉnh trước=None → PUSH                         stack='a'
bước 2: ch='b', đỉnh trước=   a → PUSH                         stack='ab'
bước 3: ch='B', đỉnh trước=   b → POP  ('b' + 'B' là cặp xấu)  stack='a'
bước 4: ch='A', đỉnh trước=   a → POP  ('a' + 'A' là cặp xấu)  stack=''
bước 5: ch='c', đỉnh trước=None → PUSH                         stack='c'
bước 6: ch='C', đỉnh trước=   c → POP  ('c' + 'C' là cặp xấu)  stack=''

s = 'leEeetcode'
bước 1: ch='l', đỉnh trước=None → PUSH                         stack='l'
bước 2: ch='e', đỉnh trước=   l → PUSH                         stack='le'
bước 3: ch='E', đỉnh trước=   e → POP  ('e' + 'E' là cặp xấu)  stack='l'
bước 4: ch='e', đỉnh trước=   l → PUSH                         stack='le'
bước 5: ch='e', đỉnh trước=   e → PUSH                         stack='lee'
bước 6: ch='t', đỉnh trước=   e → PUSH                         stack='leet'
bước 7: ch='c', đỉnh trước=   t → PUSH                         stack='leetc'
bước 8: ch

`enumerate(s, start=1)` đánh số bước từ 1 thay vì 0. `{x!s:>4}` in `x` dạng chuỗi, căn phải độ rộng 4 (để `None` và ký tự thẳng cột).

Với `"leEeetcode"`: ở bước 3, `E` khử `e`; bước 4, `e` gặp đỉnh `l` → push. Kết quả `"leetcode"` ✓.

## 7. Các lời giải khác

### 7.1. Cách 3 — Hai con trỏ, ghi đè tại chỗ

Dùng list ký tự `chars` và biến `top` = độ dài phần "stack" ở đầu list. Thay `append` bằng `chars[top] = ch; top += 1`, thay `pop` bằng `top -= 1`. Đây là cách viết stack **không cấp phát thêm** (hữu ích trong C/Java; trong Python vẫn cần list từ chuỗi).

In [12]:
def make_good_two_pointers(s: str) -> str:
    chars = list(s)
    top = 0                                  # chars[0:top] là stack
    for ch in s:
        if top > 0 and abs(ord(chars[top - 1]) - ord(ch)) == 32:
            top -= 1                         # pop
        else:
            chars[top] = ch                  # push
            top += 1
    return "".join(chars[:top])

print(repr(make_good_two_pointers("leEeetcode")))
print(repr(make_good_two_pointers("abBAcC")))

'leetcode'
''


Ở đây ta dùng điều kiện `abs(ord(x) - ord(y)) == 32` (mục 2.2) để thấy thêm một cách kiểm tra. An toàn vì input chỉ có chữ cái — với ký tự tùy ý (ví dụ `'@'` = 64 và `'`'` = 96) điều kiện này sẽ sai!

Vì sao ghi đè an toàn? `top` luôn `<=` chỉ số của `ch` đang đọc, nên ta chỉ ghi vào ô **đã đọc xong**.

### 7.2. Cách 4 — Đệ quy (minh họa)

Tìm cặp xấu đầu tiên, xóa, gọi lại hàm với chuỗi mới. Cùng độ phức tạp `O(n²)` với mô phỏng, thêm chi phí ngăn xếp gọi hàm. Chỉ để thấy **cùng một ý** có thể diễn đạt bằng đệ quy.

In [13]:
def make_good_recursive(s: str) -> str:
    for i in range(len(s) - 1):
        if s[i].swapcase() == s[i + 1]:
            return make_good_recursive(s[:i] + s[i + 2:])
    return s                                   # không còn cặp xấu → điểm dừng

print(repr(make_good_recursive("abBAcC")))

''


### 7.3. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Mô phỏng | tìm–xóa–quét lại | `O(n²)` | `O(n)` | đúng nghĩa đề, dễ nghĩ |
| 2. **Stack** | chỉ so với đỉnh | **`O(n)`** | `O(n)` | **nên dùng** |
| 3. Hai con trỏ | stack trong chính list | `O(n)` | `O(n)` (list ký tự) | mẹo tiết kiệm bộ nhớ ở ngôn ngữ khác |
| 4. Đệ quy | xóa rồi gọi lại | `O(n²)` | `O(n)` stack gọi hàm | minh họa |

## 8. Code nộp LeetCode

In [14]:
class Solution:
    def makeGood(self, s: str) -> str:
        stack = []
        for ch in s:
            if stack and stack[-1].swapcase() == ch:
                stack.pop()
            else:
                stack.append(ch)
        return "".join(stack)

sol = Solution()
for s in ["leEeetcode", "abBAcC", "s"]:
    print(repr(s), "->", repr(sol.makeGood(s)))

'leEeetcode' -> 'leetcode'
'abBAcC' -> ''
's' -> 's'


## 9. Vì sao đúng?

**Invariant:** sau khi xử lý `s[0..i]`, `stack` là **chuỗi tốt** thu được từ `s[0..i]` bằng các phép xóa hợp lệ.

- Ban đầu stack rỗng — tốt ✓.
- Thêm `ch`: nếu đỉnh và `ch` không xấu, push → chuỗi mới vẫn tốt (cặp kề mới duy nhất là (đỉnh, `ch`) và nó không xấu) ✓.
- Nếu đỉnh và `ch` xấu: xóa cặp đó là **một phép xóa hợp lệ** trong chuỗi `stack + ch`; phần còn lại `stack[:-1]` là tiền tố của chuỗi tốt nên vẫn tốt ✓.

**Vì sao thứ tự xóa không quan trọng?** (đề khẳng định đáp án duy nhất). Đây giống rút gọn từ trong **nhóm tự do**: mỗi chữ thường là "phần tử", chữ hoa là "nghịch đảo". Rút gọn `xX → ε` có tính chất **hợp lưu** (confluence): hai cách xóa khác nhau luôn có thể tiếp tục để gặp lại nhau, nên dạng cuối cùng là duy nhất. Vì vậy cách stack (xóa theo một thứ tự cụ thể) cho cùng kết quả với mọi cách khác.

Ta kiểm chứng bằng thực nghiệm: thử **mọi thứ tự xóa** trên chuỗi ngắn và so với stack.

In [15]:
from functools import lru_cache

def all_final_results(s):
    @lru_cache(maxsize=None)
    def go(t):
        outs = set()
        for i in range(len(t) - 1):
            if t[i].swapcase() == t[i + 1]:
                outs |= go(t[:i] + t[i + 2:])
        return frozenset(outs) if outs else frozenset([t])
    return go(s)

for s in ["abBAcC", "aAbBAa", "abBAaA", "xXyYzZaA", "aBbAcCa"]:
    finals = all_final_results(s)
    print(f"{s!r}: mọi kết quả có thể = {set(finals)}, stack = {make_good_stack(s)!r}")
    assert finals == {make_good_stack(s)}

'abBAcC': mọi kết quả có thể = {''}, stack = ''
'aAbBAa': mọi kết quả có thể = {''}, stack = ''
'abBAaA': mọi kết quả có thể = {''}, stack = ''
'xXyYzZaA': mọi kết quả có thể = {''}, stack = ''
'aBbAcCa': mọi kết quả có thể = {'a'}, stack = 'a'


`@lru_cache` **ghi nhớ** kết quả hàm theo tham số (memoization) để không tính lại chuỗi con đã gặp. `frozenset` là `set` bất biến (dùng được làm kết quả cache), `outs |= ...` là phép **hợp** tập.

## 10. Time / Space

Gọi `n = len(s)`.

- **Stack:** mỗi ký tự được push tối đa 1 lần và pop tối đa 1 lần → tổng `O(n)` thao tác; `join` `O(n)`. **Time `O(n)`**, **Space `O(n)`** (stack tệ nhất chứa cả chuỗi, ví dụ `"abc"`).
- **Mô phỏng:** tối đa `n/2` lần xóa × `O(n)` mỗi lần → `O(n²)`.

Lập luận "mỗi phần tử vào/ra stack tối đa một lần" gọi là **phân tích khấu hao** (amortized): dù một bước có thể không pop gì, tổng số pop không vượt quá tổng số push = `n`.

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual!r}")

sol = Solution()
check("đề 1", sol.makeGood("leEeetcode"), "leetcode")
check("đề 2", sol.makeGood("abBAcC"), "")
check("đề 3", sol.makeGood("s"), "s")
check("cùng chữ cùng kiểu", sol.makeGood("aa"), "aa")
check("chữ hoa trùng", sol.makeGood("AA"), "AA")
check("khác chữ", sol.makeGood("aB"), "aB")
check("dây chuyền", sol.makeGood("abcCBA"), "")
check("Hoa trước thường", sol.makeGood("Pp"), "")
check("còn sót lại", sol.makeGood("aAa"), "a")
check("lồng nhiều tầng", sol.makeGood("kKaBbAx"), "x")

PASS đề 1: 'leetcode'
PASS đề 2: ''
PASS đề 3: 's'
PASS cùng chữ cùng kiểu: 'aa'
PASS chữ hoa trùng: 'AA'
PASS khác chữ: 'aB'
PASS dây chuyền: ''
PASS Hoa trước thường: ''
PASS còn sót lại: 'a'
PASS lồng nhiều tầng: 'x'


**Phản ví dụ cho cách "quét một lượt":** xóa cặp xấu trong **một lần quét** trái→phải mà không xem lại đỉnh sẽ bỏ sót phản ứng dây chuyền:

In [17]:
def make_good_one_pass_wrong(s):
    out = []; i = 0
    while i < len(s):
        if i + 1 < len(s) and s[i].swapcase() == s[i + 1]:
            i += 2                      # bỏ cặp
        else:
            out.append(s[i]); i += 1
    return "".join(out)

for s in ["leEeetcode", "abBA", "abBAcC"]:
    print(repr(s), "| một lượt:", repr(make_good_one_pass_wrong(s)), "| stack:", repr(make_good_stack(s)))

'leEeetcode' | một lượt: 'leetcode' | stack: 'leetcode'
'abBA' | một lượt: 'aA' | stack: ''
'abBAcC' | một lượt: 'aA' | stack: ''


`"leEeetcode"` vẫn đúng (không dây chuyền), nhưng `"abBA"` thì một lượt để lại `"aA"` — chính là cặp xấu mới lộ ra.

### 11.1. Đối chiếu các cách trên chuỗi ngẫu nhiên

Sinh chuỗi từ bảng chữ nhỏ (`"aAbB"`) để cặp xấu xuất hiện dày đặc.

In [18]:
import random
rng = random.Random(1544)
for _ in range(2000):
    s = "".join(rng.choice("aAbBc") for _ in range(rng.randint(1, 30)))
    exp = make_good_simulate(s)
    assert make_good_stack(s) == exp
    assert make_good_two_pointers(s) == exp
    assert make_good_recursive(s) == exp
    assert Solution().makeGood(s) == exp
    res = exp
    assert all(res[i].swapcase() != res[i + 1] for i in range(len(res) - 1))   # kết quả tốt
print("2000 test ngẫu nhiên: 4 cách khớp nhau, kết quả luôn là chuỗi tốt")

2000 test ngẫu nhiên: 4 cách khớp nhau, kết quả luôn là chuỗi tốt


`all(...)` trả `True` nếu **mọi** phần tử trong dãy đúng — ta dùng để kiểm tra tính chất "không còn cặp xấu".

## 12. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ so sánh `lower()` mà quên điều kiện khác nhau
```python
if stack and stack[-1].lower() == ch.lower():   # SAI: 'aa' cũng bị xóa
```

In [19]:
def make_good_wrong_lower(s):
    st = []
    for ch in s:
        if st and st[-1].lower() == ch.lower():
            st.pop()
        else:
            st.append(ch)
    return "".join(st)

print("sai:", repr(make_good_wrong_lower("aab")), "| đúng:", repr(make_good_stack("aab")))

sai: 'b' | đúng: 'aab'


**Sửa:** thêm `stack[-1] != ch`, hoặc dùng `stack[-1].swapcase() == ch`.

### Lỗi 2 — Truy cập `stack[-1]` khi stack rỗng
```python
if stack[-1].swapcase() == ch:     # IndexError ở ký tự đầu tiên
```
**Sửa:** `if stack and ...`.

### Lỗi 3 — Dùng `str` làm stack và cộng/cắt chuỗi
```python
res = ""
...
res = res[:-1]     # mỗi lần tạo chuỗi mới O(n) → tổng O(n²)
```
Vẫn ra đúng nhưng chậm hơn; list + `join` là chuẩn.

### Lỗi 4 — Sửa chuỗi trong lúc đang duyệt theo chỉ số
Xóa phần tử trong khi `for i in range(len(s) - 1)` vẫn chạy tiếp với `len` cũ → `IndexError` hoặc bỏ sót. Trong mô phỏng ở mục 3, ta `break` ngay sau khi xóa để quét lại.

In [20]:
def simulate_no_break_bug(s):
    chars = list(s)
    try:
        for i in range(len(chars) - 1):
            if chars[i].swapcase() == chars[i + 1]:
                del chars[i:i + 2]          # xóa mà vẫn duyệt theo range cũ
        return "".join(chars)
    except IndexError as e:
        return f"IndexError: {e}"

print(simulate_no_break_bug("abBAcC"))

IndexError: list index out of range


`del lst[i:j]` xóa đoạn `i..j-1` khỏi list.

## 13. Mẹo nhận dạng pattern

- "Xóa hai phần tử **kề nhau** khi chúng 'triệt tiêu', lặp đến khi không xóa được" → **stack**, so ký tự mới với **đỉnh**.
- Có **phản ứng dây chuyền** (xóa xong lộ ra cặp mới) → càng chắc chắn là stack.
- Đề nói "thứ tự không quan trọng / đáp án duy nhất" → được phép chọn thứ tự tiện nhất (trái → phải).
- Bài cùng họ: 1047 *Remove All Adjacent Duplicates In String*, 20 *Valid Parentheses*, 735 *Asteroid Collision*, 1209 *Remove All Adjacent Duplicates II*, 844 *Backspace String Compare*.

## 14. Biến thể

### 14.1. LeetCode 1047 — xóa hai ký tự kề nhau **giống hệt**
Chỉ đổi điều kiện: `stack[-1] == ch`.

In [21]:
def remove_adjacent_duplicates(s):
    st = []
    for ch in s:
        if st and st[-1] == ch:
            st.pop()
        else:
            st.append(ch)
    return "".join(st)

print(remove_adjacent_duplicates("abbaca"))   # "ca"

ca


### 14.2. Ngoặc hợp lệ — cùng tư duy "khử với đỉnh"
`(` và `)` triệt tiêu nhau giống `a` và `A`. Chuỗi hợp lệ ⇔ sau khi khử, stack rỗng (với một loại ngoặc).

In [22]:
def balanced_after_cancel(s):
    st = []
    for ch in s:
        if st and st[-1] == "(" and ch == ")":
            st.pop()
        else:
            st.append(ch)
    return not st

for s in ["(())", "(()", ")(", "()()"]:
    print(s, balanced_after_cancel(s))

(()) True
(() False
)( False
()() True


### 14.3. Xóa `k` ký tự giống nhau liên tiếp (LeetCode 1209)
Stack lưu cặp `[ký_tự, số_lần_liên_tiếp]`; khi số lần đạt `k` thì pop.

In [23]:
def remove_k_duplicates(s, k):
    st = []                                  # mỗi phần tử: [ch, count]
    for ch in s:
        if st and st[-1][0] == ch:
            st[-1][1] += 1
            if st[-1][1] == k:
                st.pop()
        else:
            st.append([ch, 1])
    return "".join(ch * cnt for ch, cnt in st)

print(remove_k_duplicates("deeedbbcccbdaa", 3))   # "aa"

aa


`ch * cnt` lặp chuỗi `cnt` lần: `"a" * 3 == "aaa"`.

## 15. Bài tập (TODO không tự chạy)

### Bài 1 — Tính tay
Chạy stack trên `"aBbAcCd"` và ghi bảng trạng thái từng bước. Kết quả là gì?

### Bài 2 — Đoán output
```python
print("Hello".swapcase(), ord("b") - ord("B"), "x".swapcase() == "X")
```

### Bài 3 — TODO: tự viết lời giải stack

In [24]:
def exercise_make_good(s: str) -> str:
    # TODO 1: stack rỗng
    # TODO 2: với mỗi ch, nếu stack không rỗng và đỉnh là ch đổi hoa/thường → pop
    # TODO 3: ngược lại push; cuối cùng join
    pass

# assert exercise_make_good("leEeetcode") == "leetcode"
# assert exercise_make_good("abBAcC") == ""

### Bài 4 — TODO: đếm số lần xóa
Trả về **số cặp** bị xóa trong quá trình làm tốt chuỗi.

In [25]:
def exercise_count_removals(s: str) -> int:
    # TODO: mỗi lần pop là một cặp bị xóa
    pass

# assert exercise_count_removals("abBAcC") == 3
# assert exercise_count_removals("leEeetcode") == 1

### Bài 5 — TODO: Backspace (LeetCode 844 dạng đơn giản)
Ký tự `#` nghĩa là xóa ký tự ngay trước nó (nếu có). Trả về chuỗi cuối cùng.

In [26]:
def exercise_apply_backspace(s: str) -> str:
    # TODO: '#' → pop nếu stack không rỗng; ký tự khác → push
    pass

# assert exercise_apply_backspace("ab#c") == "ac"
# assert exercise_apply_backspace("a##c") == "c"
# assert exercise_apply_backspace("#a#") == ""

### Bài 6 — Mini challenge: chuỗi **dài nhất** đã tốt sẵn
Tìm độ dài đoạn con liên tiếp dài nhất của `s` mà **bản thân nó đã là chuỗi tốt** (không có cặp xấu kề nhau), trong `O(n)`.

In [27]:
def exercise_longest_good_substring(s: str) -> int:
    # TODO: quét một lượt, độ dài hiện tại reset khi gặp cặp xấu kề nhau
    pass

# assert exercise_longest_good_substring("leEeetcode") == 7
# assert exercise_longest_good_substring("aA") == 1

## 16. Gợi ý 3 mức

**Bài 1:** (1) Push/pop theo đỉnh. (2) `bB` khử trước, rồi `aA`, rồi `cC`. (3) Còn lại một ký tự.

**Bài 2:** (1) `swapcase` đổi từng ký tự. (2) Chênh mã chữ thường – hoa là hằng số. (3) `"hELLO"`, `32`, `True`.

**Bài 3:** (1) Stack chứa phần đã sạch. (2) `stack and stack[-1].swapcase() == ch`. (3) Xem mục 5.

**Bài 4:** (1) Không cần dựng kết quả. (2) Biến đếm `removed`. (3) `removed += 1` cạnh `pop()`.

**Bài 5:** (1) `#` giống "triệt tiêu với đỉnh". (2) `if ch == "#": if st: st.pop()`. (3) Chú ý `#` khi stack rỗng thì bỏ qua.

**Bài 6:** (1) Đoạn tốt bị "cắt" tại mỗi vị trí `i` mà `s[i-1], s[i]` xấu. (2) Giữ `cur` = độ dài đoạn tốt kết thúc tại `i`. (3) Xấu → `cur = 1`, không → `cur += 1`; lấy max.

## 17. Đáp án tham khảo

**Bài 1:** `a`→push `[a]`; `B`→push `[a,B]`; `b`→khử `B` `[a]`; `A`→khử `a` `[]`; `c`→push `[c]`; `C`→khử `[]`; `d`→push `[d]` → kết quả `"d"`.

**Bài 2:** `hELLO 32 True`.

**Bài 6:** với `"leEeetcode"`, các cặp xấu kề nhau là `eE` (vị trí 1–2) và `Ee` (2–3). Đoạn tốt dài nhất bắt đầu từ vị trí 3: `"eetcode"` dài 7.

In [28]:
def answer_make_good(s):
    st = []
    for ch in s:
        if st and st[-1].swapcase() == ch:
            st.pop()
        else:
            st.append(ch)
    return "".join(st)

def answer_count_removals(s):
    st = []; removed = 0
    for ch in s:
        if st and st[-1].swapcase() == ch:
            st.pop(); removed += 1
        else:
            st.append(ch)
    return removed

def answer_apply_backspace(s):
    st = []
    for ch in s:
        if ch == "#":
            if st:
                st.pop()
        else:
            st.append(ch)
    return "".join(st)

def answer_longest_good_substring(s):
    best = cur = 1
    for i in range(1, len(s)):
        cur = 1 if s[i - 1].swapcase() == s[i] else cur + 1
        best = max(best, cur)
    return best

print("Bài 2:", "Hello".swapcase(), ord("b") - ord("B"), "x".swapcase() == "X")
assert answer_make_good("aBbAcCd") == "d"
assert answer_make_good("leEeetcode") == "leetcode"
assert answer_make_good("abBAcC") == ""
assert answer_count_removals("abBAcC") == 3
assert answer_count_removals("leEeetcode") == 1
assert answer_apply_backspace("ab#c") == "ac"
assert answer_apply_backspace("a##c") == "c"
assert answer_apply_backspace("#a#") == ""
assert answer_longest_good_substring("leEeetcode") == 7
assert answer_longest_good_substring("aA") == 1
rng = random.Random(3)
for _ in range(300):     # đối chiếu bài 6 với brute force
    s = "".join(rng.choice("aAb") for _ in range(rng.randint(1, 15)))
    brute = max(j - i for i in range(len(s)) for j in range(i + 1, len(s) + 1)
                if all(s[t].swapcase() != s[t + 1] for t in range(i, j - 1)))
    assert answer_longest_good_substring(s) == brute
print("Tất cả đáp án PASS")

Bài 2: hELLO 32 True
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def makeGood(self, s: str) -> str:
        stack = []
        for ch in s:
            if stack and stack[-1].swapcase() == ch:   # đỉnh + ch là cặp xấu
                stack.pop()
            else:
                stack.append(ch)
        return "".join(stack)
```

**Câu thần chú:** *"Triệt tiêu với hàng xóm, có dây chuyền → stack, chỉ so với đỉnh."*

**Checklist trước khi submit**
- [ ] Kiểm tra `stack` khác rỗng trước khi đọc `stack[-1]`.
- [ ] Điều kiện xấu loại trừ `aa`/`AA` (dùng `swapcase` hoặc thêm `!=`).
- [ ] Test phản ứng dây chuyền (`"abBA"`) và kết quả rỗng.
- [ ] Trả về `"".join(stack)`, không trả list.
- [ ] Đúng tên method `makeGood`.